#  Task 4: News Topic Classifier Using BERT
**DevelopersHub Corp — AI/ML Internship**

---

##  Problem Statement

Automatically classifying news articles into topic categories is a fundamental NLP task.
We fine-tune a **BERT-style Transformer model** on the AG News dataset to classify
headlines into 4 categories: **World, Sports, Business, Sci/Tech**.

This task introduces:
- **Transfer Learning** — using pre-trained transformer weights and adapting them
- **Tokenization** — converting text into input IDs, attention masks, token type IDs
- **Fine-tuning** — training the full model end-to-end on downstream task data
- **Deployment** — building a live Gradio web interface for real-time classification

##  Dataset — AG News

| Property | Detail |
|---|---|
| **Source** | Hugging Face Datasets (`ag_news`) / Kaggle |
| **Total Samples** | 127,600 (train: 120,000 / test: 7,600) |
| **Classes** | World (0), Sports (1), Business (2), Sci/Tech (3) |
| **Input** | News headline text |
| **Balance** | Perfectly balanced — 30,000 per class |

##  Model Architecture

```
Input Text
    ↓
Tokenizer (bert-base-uncased)
    ↓
[CLS] + Token IDs + [SEP] + [PAD]
    ↓
BERT Encoder (12 layers, 768-dim, 12 heads)
    ↓
[CLS] Representation (768-dim)
    ↓
Dropout → Linear(768→4)
    ↓
Softmax → Class Probabilities
```

---

##  Install & Import Libraries

In [ ]:
# Install required packages
# !pip install transformers datasets gradio accelerate torch scikit-learn

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, re, math, warnings
warnings.filterwarnings('ignore')

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import load_dataset, Dataset
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)

LABELS = ['World', 'Sports', 'Business', 'Sci/Tech']
MODEL_NAME = 'bert-base-uncased'
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 2e-5

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print(f'Device          : {"cuda" if torch.cuda.is_available() else "cpu"}')

##  Load & Explore the AG News Dataset

In [ ]:
# Load AG News from Hugging Face
dataset = load_dataset('ag_news')

print('Dataset structure:')
print(dataset)
print(f'\nTrain size : {len(dataset["train"]):,}')
print(f'Test size  : {len(dataset["test"]):,}')
print(f'\nLabel names: {dataset["train"].features["label"].names}')

In [ ]:
# View sample headlines per category
df = dataset['train'].to_pandas()
print('Sample headlines per class:\n')
for i, name in enumerate(LABELS):
    sample = df[df['label']==i]['text'].iloc[0]
    print(f'[{name:10s}] {sample[:90]}...')

In [ ]:
# Class distribution
counts = df['label'].value_counts().sort_index()
print('Class distribution:')
for i, (cnt, name) in enumerate(zip(counts, LABELS)):
    bar = '█' * int(cnt/500)
    print(f'  {name:10s} | {cnt:6,} | {bar}')

##  Tokenization & Preprocessing

> BERT uses WordPiece tokenization. Every sequence starts with `[CLS]` and ends with `[SEP]`.
> `attention_mask` tells the model which tokens are real (1) vs padding (0).

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Show tokenization of a sample
sample = 'Apple unveils new AI-powered iPhone with neural processing chip'
encoding = tokenizer(sample, truncation=True, max_length=MAX_LEN, padding='max_length')

print(f'Input text    : {sample}')
print(f'\nInput IDs     : {encoding["input_ids"][:15]}...')
print(f'Attention mask: {encoding["attention_mask"][:15]}...')
print(f'\nTokens        : {tokenizer.convert_ids_to_tokens(encoding["input_ids"])[:12]}...')
print(f'\nTotal length  : {len(encoding["input_ids"])} (padded to MAX_LEN={MAX_LEN})')

In [ ]:
# Tokenize the full dataset
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LEN,
        padding='max_length'
    )

# Use a subset for faster training (use full for best results)
train_subset = dataset['train'].shuffle(seed=42).select(range(8000))
test_subset  = dataset['test'].shuffle(seed=42).select(range(2000))

train_tokenized = train_subset.map(tokenize_function, batched=True)
test_tokenized  = test_subset.map(tokenize_function, batched=True)

train_tokenized = train_tokenized.rename_column('label', 'labels')
test_tokenized  = test_tokenized.rename_column('label', 'labels')

train_tokenized.set_format('torch', columns=['input_ids','attention_mask','labels'])
test_tokenized.set_format('torch',  columns=['input_ids','attention_mask','labels'])

print(f'Tokenized train: {len(train_tokenized)} samples')
print(f'Tokenized test : {len(test_tokenized)} samples')

##  Load BERT & Fine-Tune

> `bert-base-uncased` has 110M parameters: 12 transformer layers, 768 hidden dims,
> 12 attention heads. We add a classification head on top of the `[CLS]` token output.

In [ ]:
# Load pre-trained BERT with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label={i: l for i,l in enumerate(LABELS)},
    label2id={l: i for i,l in enumerate(LABELS)}
)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params= sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Architecture        : {MODEL_NAME}')

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1_macro': float(f1_score(labels, preds, average='macro'))
    }

training_args = TrainingArguments(
    output_dir            = './bert_news_checkpoints',
    num_train_epochs      = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 64,
    evaluation_strategy   = 'epoch',
    save_strategy         = 'epoch',
    load_best_model_at_end= True,
    metric_for_best_model = 'f1_macro',
    learning_rate         = LR,
    weight_decay          = 0.01,
    warmup_ratio          = 0.1,
    logging_steps         = 100,
    report_to             = 'none',
    fp16                  = torch.cuda.is_available(),  # use FP16 on GPU
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_tokenized,
    eval_dataset  = test_tokenized,
    tokenizer     = tokenizer,
    data_collator = DataCollatorWithPadding(tokenizer),
    compute_metrics= compute_metrics,
)

print('Starting fine-tuning...')
trainer.train()
print('\nFine-tuning complete ')

##  Evaluation

Expected results with full dataset (3 epochs on GPU):
| Metric | Typical Score |
|---|---|
| Accuracy | ~94–95% |
| F1 Macro | ~0.94–0.95 |
| Per-class F1 | ~0.93–0.96 per class |

In [ ]:
eval_results = trainer.evaluate()
print(f'Accuracy : {eval_results["eval_accuracy"]:.4f} ({eval_results["eval_accuracy"]*100:.2f}%)')
print(f'F1 Macro : {eval_results["eval_f1_macro"]:.4f}')

preds_output = trainer.predict(test_tokenized)
preds  = np.argmax(preds_output.predictions, axis=-1)
labels = test_tokenized['labels'].numpy()

print('\n--- Classification Report ---')
print(classification_report(labels, preds, target_names=LABELS))

### Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(labels, preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', ax=ax,
            cmap='Blues', xticklabels=LABELS, yticklabels=LABELS,
            linewidths=1, annot_kws={'size':13})
ax.set_title(f'Confusion Matrix — Accuracy: {eval_results["eval_accuracy"]:.2%}', fontsize=13)
ax.set_xlabel('Predicted'); ax.set_ylabel('True Label')
plt.tight_layout(); plt.show()

### Per-Class F1 Score

In [ ]:
per_f1 = f1_score(labels, preds, average=None)
fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(LABELS, per_f1, color=['#4ECDC4','#FF6B6B','#F5D547','#45B7D1'],
              alpha=0.85, edgecolor='none', width=0.5)
for bar, val in zip(bars, per_f1):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.005, f'{val:.3f}',
            ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.set_title(f'Per-Class F1 Score | Macro Avg: {f1_score(labels,preds,average="macro"):.4f}')
ax.set_ylabel('F1 Score')
plt.tight_layout(); plt.show()

##  Save the Model

In [ ]:
# Save fine-tuned model and tokenizer
model.save_pretrained('./bert_news_model')
tokenizer.save_pretrained('./bert_news_model')
print('Model saved to ./bert_news_model ')
print('Files saved:')
import os
for f in os.listdir('./bert_news_model'): print(f'  {f}')

##  Gradio Deployment — Live Demo

> Run this cell to launch an interactive web interface for real-time classification.
> Share the public URL with anyone — no server needed.

In [ ]:
import gradio as gr
from transformers import pipeline

# Load saved model as pipeline
classifier = pipeline(
    'text-classification',
    model='./bert_news_model',
    tokenizer='./bert_news_model',
    return_all_scores=True
)

def classify_news(headline):
    if not headline.strip():
        return {l: 0.0 for l in LABELS}
    results = classifier(headline[:512])[0]
    scores  = {r['label']: round(r['score'], 4) for r in results}
    return scores

examples = [
    'Apple unveils new AI-powered chip that beats Nvidia on benchmarks',
    'Manchester City defeats Real Madrid 3-1 in Champions League final',
    'Fed raises interest rates amid rising inflation concerns',
    'UN Security Council calls for immediate ceasefire in Gaza',
    'NASA Artemis mission successfully lands astronauts on lunar surface',
    'Tesla reports record Q3 earnings, stock surges 12%',
]

demo = gr.Interface(
    fn=classify_news,
    inputs=gr.Textbox(
        label='News Headline',
        placeholder='Enter a news headline to classify...',
        lines=3
    ),
    outputs=gr.Label(
        label='Topic Classification',
        num_top_classes=4
    ),
    title='📰 News Topic Classifier (BERT)',
    description=(
        'Fine-tuned BERT model classifying news headlines into: '
        'World, Sports, Business, or Sci/Tech. '
        'Trained on the AG News dataset using Hugging Face Transformers.'
    ),
    examples=[[e] for e in examples],
    theme=gr.themes.Soft(),
    allow_flagging='never'
)

demo.launch(share=True)  # share=True gives a public URL
# demo.launch()         # local only

##  Results & Key Insights

###  Expected Performance (bert-base-uncased, full dataset, 3 epochs)

| Metric | Score |
|---|---|
| Test Accuracy | ~94–95% |
| F1 Macro | ~0.94–0.95 |
| World F1 | ~0.93 |
| Sports F1 | ~0.98 |
| Business F1 | ~0.92 |
| Sci/Tech F1 | ~0.94 |

---

###  Key Findings

**1. Why BERT works so well for text classification:**
- BERT is pre-trained on 3.3B words (Wikipedia + BooksCorpus)
- It already understands word meaning, context, and relationships
- Fine-tuning just adapts its `[CLS]` representation to our 4-class task
- Only 3 epochs needed — the heavy lifting was done in pre-training

**2. Transfer Learning advantage:**
- Training BERT from scratch would need millions of examples and days of compute
- Fine-tuning on 8,000 examples takes ~15 minutes on a T4 GPU and reaches 94%+
- This is the power of **transfer learning** — apply knowledge from one task to another

**3. Tokenization matters:**
- BERT uses WordPiece — rare words split into subwords (e.g., 'unaffected' → ['un','##affected'])
- `[CLS]` token aggregates the whole sequence for classification
- Attention masks ensure padding tokens don't affect predictions

**4. Class difficulty:**
- Sports headlines are easiest to classify (domain-specific vocabulary)
- Business vs World sometimes overlaps (geopolitical + economic news)
- Sci/Tech vs Business can confuse (tech company earnings news)

**5. Deployment with Gradio:**
- `demo.launch(share=True)` creates a public HTTPS URL in seconds
- No server, no Docker, no cloud setup needed
- The `pipeline()` API makes inference in 2 lines of code

---

###  Limitations

- BERT max input is 512 tokens — full articles need chunking or DistilBERT
- Model is English-only (`bert-base-uncased`)
- For production, consider `distilbert-base-uncased` (40% smaller, 97% of accuracy)

---
*Task 4 Complete — DevelopersHub Corp ML Internship*